# Uni-Hema (CVPR 2026) — run `app.py` on Google Colab

Run the cells **one at a time, top to bottom**. Before you start: **Runtime → Change runtime type → T4 GPU** (the code calls `.cuda()` so a GPU is required).

| Step | What it does |
|---|---|
| 1 | Check GPU + versions |
| 2 | Clone the GitHub repo |
| 3 | Install the missing packages (keeps Colab's own PyTorch) |
| 4 | Apply version-compatibility patches |
| 5 | (Optional) compile the CUDA deformable-attention operator |
| 6 | Download the trained Uni-Hema checkpoint (1.2 GB) |
| 7 | Load the model |
| 8 | Test all 6 tasks inside the notebook |
| 9 | Launch the Gradio web app |

## Step 1 — Check GPU and versions

In [ ]:
!nvidia-smi
import sys, torch, torchvision
print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torchvision :", torchvision.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then run again."

## Step 2 — Get the code from GitHub

In [ ]:
%cd /content
!rm -rf Uni-Hema
!git clone -q https://github.com/intelligentMachines-ITU/Uni-Hema.git
%cd /content/Uni-Hema
!ls

*Alternative (if GitHub fails):* upload your `Uni-Hema-main.zip` with the Files panel and run
`!unzip -q /content/Uni-Hema-main.zip -d /content && mv /content/Uni-Hema-main /content/Uni-Hema`

## Step 3 — Install packages
We **do not reinstall PyTorch** (Colab's version is fine and reinstalling it is what usually breaks things).
We pin `transformers` to 4.x because the model code uses T5 APIs that were removed in transformers 5.

In [ ]:
!pip install -q "transformers==4.46.3" sentencepiece protobuf "gradio>=5,<6" \
    addict yapf termcolor fvcore pycocotools timm evaluate rouge_score
import importlib.metadata as md
for p in ["transformers", "tokenizers", "huggingface_hub", "gradio"]:
    print(f"{p:16s}", md.version(p))

## Step 4 — Apply compatibility patches
Fixes, in order:
1. deletes stale `__pycache__` / `build` folders shipped in the repo (compiled for other Python versions)
2. removes `from transformers import AdamW` (removed from new transformers)
3. updates the old CUDA C++ calls (`.type()`, `.data<T>()`) so the operator compiles with PyTorch 2.x
4. adds a pure-PyTorch fallback for deformable attention, so the model runs even if compiling fails
5. `app.py`: `torch.load(..., weights_only=False)` (PyTorch ≥ 2.6 change), prints missing/unexpected keys, font fallback, `torch.no_grad()` for inference, full error tracebacks, and no auto-launch on import

In [ ]:
%%writefile /content/patch_unihema.py
import os, re, glob, shutil

REPO = os.environ.get("UNIHEMA_REPO", "/content/Uni-Hema")

def edit(path, pairs, required=True):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    for old, new in pairs:
        if new in s:
            continue  # already patched (safe to re-run this cell)
        elif old in s:
            s = s.replace(old, new)
        elif required:
            raise RuntimeError(f"Pattern not found in {path}:\n{old[:80]}")
    with open(path, "w", encoding="utf-8") as f:
        f.write(s)

# 0) Remove stale compiled files that came with the repo (built for other Python versions)
for p in glob.glob(f"{REPO}/**/__pycache__", recursive=True):
    shutil.rmtree(p, ignore_errors=True)
shutil.rmtree(f"{REPO}/models/dino/ops/build", ignore_errors=True)
print("✔ removed stale __pycache__ / build folders")

# 1) transformers.AdamW no longer exists in new transformers versions
n = 0
for py in glob.glob(f"{REPO}/**/*.py", recursive=True):
    with open(py, encoding="utf-8", errors="ignore") as f:
        s = f.read()
    s2 = s.replace("T5ForConditionalGeneration, AdamW", "T5ForConditionalGeneration")
    if s2 != s:
        with open(py, "w", encoding="utf-8") as f:
            f.write(s2)
        n += 1
print(f"✔ removed 'AdamW' import from transformers in {n} files")

# 2) Old CUDA C++ API (.type(), .data<T>()) does not compile on PyTorch 2.x
src = f"{REPO}/models/dino/ops/src"
for f in glob.glob(f"{src}/**/*.*", recursive=True):
    if not f.endswith((".h", ".cu", ".cuh", ".cpp")):
        continue
    s = open(f).read()
    s2 = s.replace(".type().is_cuda()", ".is_cuda()")
    s2 = re.sub(r"AT_DISPATCH_FLOATING_TYPES\((\w+)\.type\(\)", r"AT_DISPATCH_FLOATING_TYPES(\1.scalar_type()", s2)
    s2 = re.sub(r"\.data<(\w+)>\(\)", r".data_ptr<\1>()", s2)
    if s2 != s:
        open(f, "w").write(s2)
print("✔ modernised CUDA operator source for PyTorch 2.x")

# 3) Pure-PyTorch fallback if the CUDA operator is not compiled
edit(f"{REPO}/models/dino/ops/functions/ms_deform_attn_func.py", [(
"import MultiScaleDeformableAttention as MSDA\n",
"""try:
    import MultiScaleDeformableAttention as MSDA
    MSDA_AVAILABLE = True
except ImportError:
    MSDA = None
    MSDA_AVAILABLE = False
    print("[Uni-Hema] CUDA op MultiScaleDeformableAttention not found -> using slower pure-PyTorch fallback")
""")])
with open(f"{REPO}/models/dino/ops/functions/ms_deform_attn_func.py", "a") as f:
    if "_MSDeformAttnDispatch" not in open(f.name).read():
        f.write('''

# ---- added for Colab: use CUDA kernel when available, otherwise pure PyTorch (autograd-safe) ----
_MSDeformAttnCUDA = MSDeformAttnFunction

class _MSDeformAttnDispatch:
    @staticmethod
    def apply(value, value_spatial_shapes, value_level_start_index, sampling_locations, attention_weights, im2col_step):
        if MSDA_AVAILABLE and value.is_cuda:
            return _MSDeformAttnCUDA.apply(value, value_spatial_shapes, value_level_start_index,
                                           sampling_locations, attention_weights, im2col_step)
        return ms_deform_attn_core_pytorch(value, value_spatial_shapes, sampling_locations, attention_weights)

MSDeformAttnFunction = _MSDeformAttnDispatch
''')
# fallback must accept spatial shapes given as a tensor
edit(f"{REPO}/models/dino/ops/functions/ms_deform_attn_func.py", [(
"""    N_, S_, M_, D_ = value.shape
    _, Lq_, M_, L_, P_, _ = sampling_locations.shape
""",
"""    N_, S_, M_, D_ = value.shape
    _, Lq_, M_, L_, P_, _ = sampling_locations.shape
    value_spatial_shapes = [(int(H_), int(W_)) for H_, W_ in value_spatial_shapes]
""")])
print("✔ added pure-PyTorch fallback for deformable attention")

# 4) app.py fixes
edit(f"{REPO}/app.py", [
    # PyTorch >= 2.6 defaults to weights_only=True, which cannot open this checkpoint
    ('checkpoint = torch.load(checkpoint_path, map_location="cpu")\nmodel.load_state_dict(checkpoint["model"], strict=False)',
     'checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n'
     'load_info = model.load_state_dict(checkpoint["model"], strict=False)\n'
     'print("Missing keys   :", len(load_info.missing_keys), load_info.missing_keys[:10])\n'
     'print("Unexpected keys:", len(load_info.unexpected_keys), load_info.unexpected_keys[:10])\n'
     'del checkpoint'),
    # font may not exist -> fall back to default font
    ('    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)',
     '    try:\n        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)\n'
     '    except OSError:\n        font = ImageFont.load_default()'),
    # no gradients needed for inference (saves a lot of GPU memory)
    ('def dino_inference(pil_image,task, prompt):',
     '@torch.no_grad()\ndef dino_inference(pil_image,task, prompt):'),
    # show the full error in the Colab output instead of hiding it
    ('    except Exception as e:\n        return None, f"Error: {str(e)}"',
     '    except Exception as e:\n        import traceback; traceback.print_exc()\n        return None, f"Error: {str(e)}"'),
    # label 0 is not in id2name -> KeyError '0'; use a safe lookup
    ("label_names = [id2name[str(int(l))] for l in labels]",
     'label_names = [id2name.get(str(int(l)), "unknown") for l in labels]'),
    # QA / MLM answer comes back as a list -> show plain text
    ("            text_output = outputs['pred_text'] # shape: (B, 64, 64)",
     "            text_output = outputs['pred_text']\n"
     "            if isinstance(text_output, (list, tuple)): text_output = \" \".join(map(str, text_output))"),
    # don't auto-launch on import; we launch from the notebook with a public link
    ('\ndemo.launch()', '\nif __name__ == "__main__":\n    demo.launch(share=True, debug=True)'),
])
print("✔ patched app.py (torch.load, font, no_grad, traceback, launch)")
print("\nAll patches applied.")

In [ ]:
!python /content/patch_unihema.py

## Step 4b — QA fixes (run after Step 4, before Step 7)
1. `app.py`: the Question box was locked (`interactive=False`), so you couldn't type your own question.
2. `dino.py`: `generate()` was only given the text tokens, so QA answers ignored the image. It now gets the image-conditioned encoder output.

Safe to re-run. If you re-clone the repo, run Step 4 and this cell again.

In [ ]:
import re

# ---- Patch A: app.py -> QA question box editable (and cleared on task change) ----
p = "/content/Uni-Hema/app.py"
s = open(p).read()
old = """    elif task == "Question Answering":
        text_update = gr.update(
            visible=True,
            interactive=False,
            label="Question"
        )"""
new = """    elif task == "Question Answering":
        text_update = gr.update(
            value="",
            visible=True,
            interactive=True,
            label="Question",
            placeholder="Q: What is the texture of the cytoplasm?"
        )"""
if new in s:
    print("Patch A already applied")
else:
    assert old in s, "QA block not found - run the Step 4 patch cell first"
    open(p, "w").write(s.replace(old, new))
    print("Patch A applied: QA question box is editable")

# ---- Patch B: dino.py -> generate() uses image-conditioned encoder output ----
p = "/content/Uni-Hema/models/dino/dino.py"
s = open(p).read()
if "encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states)" in s:
    print("Patch B already applied")
else:
    i = s.index("def decode_from_encoder")
    head, tail = s[:i], s[i:]
    tail, n = re.subn(
        r"input_ids=encoder_input_ids\.to\(device\),[^\n]*",
        "encoder_outputs=BaseModelOutput(last_hidden_state=encoder_hidden_states),",
        tail, count=1)
    assert n == 1, "generate line not found"
    s = "from transformers.modeling_outputs import BaseModelOutput\n" + head + tail
    open(p, "w").write(s)
    print("Patch B applied: generate() now uses image-conditioned encoder output")

## Step 5 — (Optional but recommended) compile the CUDA operator
Takes ~3–6 minutes. **If this fails, don't worry** — Step 4 added a pure-PyTorch fallback, the app still works (just a bit slower).

In [ ]:
import os, torch
major, minor = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"   # compile only for your GPU -> faster
!nvcc --version | tail -2
%cd /content/Uni-Hema/models/dino/ops
!pip install . --no-build-isolation 2>&1 | tail -5
%cd /content/Uni-Hema
!python -c "import MultiScaleDeformableAttention; print('✅ CUDA operator compiled and importable')" || echo "⚠️ Not compiled -> fallback will be used (that's OK)"

## Step 6 — Download the trained model from Hugging Face (≈1.2 GB)

In [ ]:
from huggingface_hub import hf_hub_download
config_path = hf_hub_download("ryhm/Uni-hema", "config_args_all.json")
checkpoint_path = hf_hub_download("ryhm/Uni-hema", "checkpoint0023.pth")
print(config_path); print(checkpoint_path)

## Step 7 — Load the model
This imports `app.py`, which builds the model, downloads `google/flan-t5-base`, and loads the checkpoint.
Check the printed **Missing / Unexpected keys**. A small number of missing keys is normal with `strict=False`; hundreds would mean the weights didn't match.

In [ ]:
%cd /content/Uni-Hema
import sys, os
sys.path.insert(0, "/content/Uni-Hema")
import app
print("Model on:", next(app.model.parameters()).device)

## Step 8 — Test every task inside the notebook (no web UI)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt, numpy as np

tests = [
    ("Detection",                 "sample_det/193.jpg",              ""),
    ("Detection with Morphology", "sample_dm/18_33_1000_ALL.png",    ""),
    ("Segmentation",              "sample_seg/10.jpg",               ""),
    ("Classification",            "sample_cls/95-8-6-1_61_2.jpg",    ""),
    ("Question Answering",        "sample_qa/EO_85775.jpg",          "Q: What can you observe in the image of the cell?"),
    ("Masked Language Modeling",  "sample_mlm/22_1_1000_AML.png",
     "Mask:Myeloblasts are <extra_id_0> in number, suggesting a possible hematological malignancy."),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (task, path, prompt) in zip(axes.ravel(), tests):
    out_img, text = app.dino_inference(Image.open(path), task, prompt)
    print(f"[{task}] {text}")
    if out_img is not None:
        ax.imshow(out_img, cmap="gray")
    ax.set_title(task); ax.axis("off")
plt.tight_layout(); plt.show()

## Step 9 — Launch the Gradio app (same UI as `app.py`)
Click the **public URL** (`https://xxxx.gradio.live`). The cell keeps running while the app is up. Press **Stop** on the cell to shut it down.

In [ ]:
app.demo.launch(share=True, debug=True)